# ListenBrainz Listen Cleaning

Inspect the supplied JSON-lines schema and run a bounded 10,000-line cleaning preview. The full `9.listens` dump is several gigabytes, so this notebook does not load the entire file into memory.

In [3]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

PROJECT_DIR = Path.cwd()
RAW_PATH = PROJECT_DIR / "9.listens"
SCRIPT_PATH = PROJECT_DIR / "clean_listens.py"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "notebook_preview"
assert RAW_PATH.is_file(), f"Missing input file: {RAW_PATH}"
assert SCRIPT_PATH.is_file(), f"Missing cleaner: {SCRIPT_PATH}"

## Inspect Source Records

Each input line is one listen. This cell reads only five records so the large dump is not loaded into memory.

In [4]:
with RAW_PATH.open(encoding="utf-8") as source:
    source_rows = [json.loads(next(source)) for _ in range(5)]
schema_preview = pd.json_normalize(source_rows, sep=".")
print("Columns found:", schema_preview.columns.tolist())

Columns found: ['user_id', 'user_name', 'timestamp', 'recording_msid', 'track_metadata.track_name', 'track_metadata.artist_name', 'track_metadata.release_name', 'track_metadata.additional_info.duration', 'track_metadata.additional_info.tracknumber', 'track_metadata.additional_info.release_mbid', 'track_metadata.additional_info.recording_mbid', 'track_metadata.additional_info.submission_client', 'track_metadata.additional_info.recording_msid', 'track_metadata.additional_info.duration_ms', 'track_metadata.additional_info.artist_mbids', 'track_metadata.additional_info.artist_names', 'track_metadata.additional_info.submission_client_version', 'track_metadata.additional_info.media_player']


The selected columns show who listened, when, and the track metadata. The source timestamp is Unix time; the cleaning script converts it to UTC.

In [5]:
source_preview = schema_preview[[
    "user_id",
    "timestamp",
    "track_metadata.artist_name",
    "track_metadata.track_name",
]]
display(source_preview)

,user_id,timestamp,track_metadata.artist_name,track_metadata.track_name
0,39653,1790545436,Radiator Hospital,Are You Feeling Me?
1,93908,1790553602,League Of Legends • Rob Robinson,Deceit
2,30220,1790553558,Die drei ???,146 - Der Biss der Bestie - Teil 36
3,137514,1790553524,CORTIS,MOTION (feat. Juicy J)
4,171240,1790553534,.｡*ﾟ+.*.𝓒𝓾𝓻𝓻𝓮𝓷｡+..｡*ﾟ+,worldwidesuperstar


## Run the Cleaner

Run the reusable script on the first 10,000 input lines. This quick preview writes separate CSVs under `outputs/notebook_preview/`; the full run is `python clean_listens.py` in a terminal.

In [11]:
subprocess.run(
    [sys.executable, str(SCRIPT_PATH), "--limit", "10000", "--output-dir", str(OUTPUT_DIR)],
    cwd=PROJECT_DIR,
    check=True,
)

Read 10,000 lines; wrote 10,000 clean rows.
Outputs: /Users/leohwang/dso576/dso576-project/outputs/notebook_preview


CompletedProcess(args=['/Users/leohwang/dso576/dso576-project/.venv/bin/python', '/Users/leohwang/dso576/dso576-project/clean_listens.py', '--limit', '10000', '--output-dir', '/Users/leohwang/dso576/dso576-project/outputs/notebook_preview'], returncode=0)

## Cleaning Summary

This table reports input lines, dropped records, duplicates, and the number of clean and review rows. It makes the cleaning impact auditable.

In [12]:
summary = pd.read_csv(OUTPUT_DIR / "cleaning_summary.csv")
display(summary)

,step,rows
0,input lines read,10000
1,rows missing user or timestamp,0
2,exact duplicate rows removed,0
3,future listens removed,0
4,clean rows written,10000
5,rows needing review,9149


## Cleaned Records

This table displays a few cleaned listens, including normalized UTC timestamps, artist identifiers, durations in milliseconds, and quality flags.

In [13]:
clean_preview = pd.read_csv(OUTPUT_DIR / "listens_clean.csv")
display(clean_preview.head())

,source_row,user_id,user_name,listened_at,recording_msid,recording_mbid,artist_mbids,primary_artist_mbid,n_credited_artists,artist_name,artist_key,track_name,release_name,duration_ms,flag_key_conflict,flag_outside_month,flag_future,flag_bad_duration,flag_heavy_user
0,1,39653,galen8183,2026-09-27T21:43:56Z,13a11b06-d55c-4eb3-a2d7-4121a15e788f,10d83f54-08bc-4812-a177-ebbdbf26f932,NaN,NaN,0,Radiator Hospital,radiator hospital,Are You Feeling Me?,Something Wild,112000.0,False,False,False,False,False
1,2,93908,snowsuit3207,2026-09-28T00:00:02Z,a4167d77-26e9-47c7-9fa9-c7ba5fddc756,NaN,0976dbd8-e7fe-4161-9a49-58c57f7a7787,0976dbd8-e7fe-4161-9a49-58c57f7a7787,1,League Of Legends • Rob Robinson,league of legends • rob robinson,Deceit,Sessions: Star Guardian Taliyah,186860.0,False,False,False,False,False
2,3,30220,miasinterestinglife,2026-09-27T23:59:18Z,5439531a-db20-4aa8-9da4-0a15da0bb6f3,NaN,NaN,NaN,0,Die drei ???,die drei ???,146 - Der Biss der Bestie - Teil 36,Die drei ??? 146: Der Biss der Bestie,87085.0,False,False,False,False,False
3,4,137514,raspybxrry,2026-09-27T23:58:44Z,1ff57c3c-c771-40e3-9d62-37550202924a,NaN,NaN,NaN,0,CORTIS,cortis,MOTION (feat. Juicy J),MOTION (feat. Juicy J) - Single,157361.0,False,False,False,False,False
4,5,171240,dR3aMe1,2026-09-27T23:58:54Z,e98fbdba-4812-4445-b26b-a12089ecb91f,NaN,NaN,NaN,0,.｡*ﾟ+.*.𝓒𝓾𝓻𝓻𝓮𝓷｡+..｡*ﾟ+,.｡*ﾟ+.*.𝓒𝓾𝓻𝓻𝓮𝓷｡+..｡*ﾟ+,worldwidesuperstar,Crown of Cuteness,128000.0,False,False,False,False,False


## Audit Evidence

These compact reports show the decisions behind the preview: quality flags, missing values, and the five individual record checks. The full run writes the same files for all source records.

In [14]:
flag_preview = pd.read_csv(OUTPUT_DIR / "data_quality_flags.csv")
missing_preview = pd.read_csv(OUTPUT_DIR / "missing_value_audit.csv")
record_check_preview = pd.read_csv(OUTPUT_DIR / "record_checks.csv")
display(flag_preview)
display(missing_preview)
display(record_check_preview[[
    "source_row",
    "expected_result_and_reason",
    "actual_disposition",
    "actual_listened_at",
    "actual_duration_ms",
    "actual_primary_artist_mbid",
    "matches_expected_disposition",
    "matches_expected_values",
]])

,flag,rows_flagged
0,flag_key_conflict,0
1,flag_outside_month,500
2,flag_future,0
3,flag_bad_duration,2
4,flag_heavy_user,2351


,column,missing_before,missing_after,handling
0,user_id,0,0,Required for listener-level analysis; rows mis...
1,user_name,0,0,Retained as missing unless required user or ti...
2,listened_at,0,0,Required for time-based analysis; rows with in...
3,recording_msid,0,0,Retained as missing; listen may still be valid...
4,recording_mbid,9327,9327,Retained as missing; external recording enrich...
5,artist_mbids,9149,9149,Retained as missing; artist name remains avail...
6,primary_artist_mbid,9149,9149,Retained as missing; row is included in review...
7,artist_name,0,0,Retained as missing unless required user or ti...
8,track_name,0,0,Retained as missing unless required user or ti...
9,release_name,432,432,Retained as missing unless required user or ti...


,source_row,expected_result_and_reason,actual_disposition,actual_listened_at,actual_duration_ms,actual_primary_artist_mbid,matches_expected_disposition,matches_expected_values
0,1,"Keep; convert timestamp to UTC, convert durati...",kept,2026-09-27T21:43:56Z,112000.0,NaN,True,True
1,2,"Keep; convert timestamp to UTC, preserve durat...",kept,2026-09-28T00:00:02Z,186860.0,0976dbd8-e7fe-4161-9a49-58c57f7a7787,True,True
2,3,"Keep; convert timestamp to UTC, preserve durat...",kept,2026-09-27T23:59:18Z,87085.0,NaN,True,True
3,4,"Keep; convert timestamp to UTC, preserve durat...",kept,2026-09-27T23:58:44Z,157361.0,NaN,True,True
4,5,"Keep; convert timestamp to UTC, preserve durat...",kept,2026-09-27T23:58:54Z,128000.0,NaN,True,True
